<a href="https://colab.research.google.com/github/nurfajrianiriri-lgtm/Tugas-4-Sistem-temu-kembali/blob/main/240210500002_Nur_fajriani_Burhan_TFIDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import library yang diperlukan
import re
import math
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Dataset 4 dokumen
dokumen = [
    "Komputer digunakan mahasiswa untuk mengerjakan tugas kuliah. Komputer membantu pekerjaan menjadi lebih cepat.",

    "Jaringan komputer menghubungkan beberapa komputer agar dapat bertukar data. Jaringan sering digunakan di kampus.",

    "Teknologi komputer terus berkembang dan semakin banyak digunakan dalam kehidupan sehari-hari. Mahasiswa menggunakan teknologi untuk belajar.",

    "Sistem komputer terdiri dari perangkat keras dan perangkat lunak. Kedua bagian tersebut bekerja bersama untuk menjalankan komputer."
]

# Membuat DataFrame
df_dokumen = pd.DataFrame({
    "Dokumen": ["D1", "D2", "D3", "D4"],
    "Teks": dokumen
})

display(df_dokumen)

,Dokumen,Teks
0,D1,Komputer digunakan mahasiswa untuk mengerjakan...
1,D2,Jaringan komputer menghubungkan beberapa kompu...
2,D3,Teknologi komputer terus berkembang dan semaki...
3,D4,Sistem komputer terdiri dari perangkat keras d...


In [ ]:
# Daftar stopword sederhana Bahasa Indonesia
stopwords = {
    "yang", "dan", "di", "ke", "dari", "untuk",
    "dengan", "dalam", "agar", "dapat", "sering",
    "menjadi", "lebih", "pada", "atau", "ini",
    "itu", "sangat", "terus", "beberapa"
}


def preprocess_text(text):
    """
    Melakukan preprocessing sederhana:
    1. Case folding
    2. Tokenisasi
    3. Stopword removal
    """

    # Case folding
    text = text.lower()

    # Tokenisasi dan menghilangkan tanda baca
    tokens = re.findall(r'\b[a-z]+\b', text)

    # Menghapus stopword
    tokens = [token for token in tokens if token not in stopwords]

    return tokens

In [ ]:
# Menerapkan preprocessing pada setiap dokumen
df_preprocessing = df_dokumen.copy()

df_preprocessing["Token"] = df_preprocessing["Teks"].apply(preprocess_text)

# Menampilkan hasil preprocessing
display(df_preprocessing[["Dokumen", "Token"]])

,Dokumen,Token
0,D1,"[komputer, digunakan, mahasiswa, mengerjakan, ..."
1,D2,"[jaringan, komputer, menghubungkan, komputer, ..."
2,D3,"[teknologi, komputer, berkembang, semakin, ban..."
3,D4,"[sistem, komputer, terdiri, perangkat, keras, ..."


In [ ]:
# Mengambil seluruh token dari semua dokumen
all_tokens = []

for tokens in df_preprocessing["Token"]:
    all_tokens.extend(tokens)

# Membuat daftar term unik
terms = sorted(set(all_tokens))

print("Jumlah term:", len(terms))
print("\nDaftar term:")
print(terms)

Jumlah term: 34

Daftar term:
['bagian', 'banyak', 'bekerja', 'belajar', 'berkembang', 'bersama', 'bertukar', 'cepat', 'data', 'digunakan', 'hari', 'jaringan', 'kampus', 'kedua', 'kehidupan', 'keras', 'komputer', 'kuliah', 'lunak', 'mahasiswa', 'membantu', 'mengerjakan', 'menggunakan', 'menghubungkan', 'menjalankan', 'pekerjaan', 'perangkat', 'sehari', 'semakin', 'sistem', 'teknologi', 'terdiri', 'tersebut', 'tugas']


In [ ]:
# Membuat matriks Bag-of-Words dengan nilai awal 0
df_bow = pd.DataFrame(
    0,
    index=df_preprocessing["Dokumen"],
    columns=terms
)

# Menghitung jumlah kemunculan setiap term pada setiap dokumen
for i, tokens in enumerate(df_preprocessing["Token"]):
    dokumen_id = df_preprocessing.loc[i, "Dokumen"]

    for term in tokens:
        df_bow.loc[dokumen_id, term] += 1

print("Matriks Bag-of-Words:")
display(df_bow)

Matriks Bag-of-Words:


,bagian,banyak,bekerja,belajar,berkembang,bersama,bertukar,cepat,data,digunakan,...,menjalankan,pekerjaan,perangkat,sehari,semakin,sistem,teknologi,terdiri,tersebut,tugas
Dokumen,,,,,,,,,,,,,,,,,,,,,
D1,0,0,0,0,0,0,0,1,0,1,...,0,1,0,0,0,0,0,0,0,1
D2,0,0,0,0,0,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
D3,0,1,0,1,1,0,0,0,0,1,...,0,0,0,1,1,0,2,0,0,0
D4,1,0,1,0,0,1,0,0,0,0,...,1,0,2,0,0,1,0,1,1,0


In [ ]:
# TF menggunakan jumlah kemunculan term dalam dokumen
# Pada bentuk ini, TF sama dengan nilai Bag-of-Words

tf_manual = df_bow.astype(float)

print("Matriks Term Frequency (TF):")
display(tf_manual)

Matriks Term Frequency (TF):


,bagian,banyak,bekerja,belajar,berkembang,bersama,bertukar,cepat,data,digunakan,...,menjalankan,pekerjaan,perangkat,sehari,semakin,sistem,teknologi,terdiri,tersebut,tugas
Dokumen,,,,,,,,,,,,,,,,,,,,,
D1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
D2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
D3,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,1.0,0.0,2.0,0.0,0.0,0.0
D4,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,2.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0


In [ ]:
# Menghitung Document Frequency (DF)
df_manual = (df_bow > 0).sum(axis=0)

df_df = pd.DataFrame({
    "Term": df_manual.index,
    "DF": df_manual.values
})

print("Document Frequency (DF):")
display(df_df)

Document Frequency (DF):


,Term,DF
0,bagian,1
1,banyak,1
2,bekerja,1
3,belajar,1
4,berkembang,1
5,bersama,1
6,bertukar,1
7,cepat,1
8,data,1
9,digunakan,3


In [ ]:
# Jumlah seluruh dokumen
N = len(df_dokumen)

# Menghitung IDF untuk setiap term
idf_manual = df_manual.apply(
    lambda df: math.log(N / df) + 1
)

df_idf = pd.DataFrame({
    "Term": idf_manual.index,
    "DF": df_manual.values,
    "IDF": idf_manual.values
})

print("DF dan IDF:")
display(df_idf.round(4))

DF dan IDF:


,Term,DF,IDF
0,bagian,1,2.3863
1,banyak,1,2.3863
2,bekerja,1,2.3863
3,belajar,1,2.3863
4,berkembang,1,2.3863
5,bersama,1,2.3863
6,bertukar,1,2.3863
7,cepat,1,2.3863
8,data,1,2.3863
9,digunakan,3,1.2877


In [ ]:
# Menghitung TF-IDF secara manual
tfidf_manual = tf_manual.multiply(idf_manual, axis=1)

print("Matriks TF-IDF Manual:")
display(tfidf_manual.round(4))

Matriks TF-IDF Manual:


,bagian,banyak,bekerja,belajar,berkembang,bersama,bertukar,cepat,data,digunakan,...,menjalankan,pekerjaan,perangkat,sehari,semakin,sistem,teknologi,terdiri,tersebut,tugas
Dokumen,,,,,,,,,,,,,,,,,,,,,
D1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863,0.0000,1.2877,...,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863
D2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863,0.0000,2.3863,1.2877,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
D3,0.0000,2.3863,0.0000,2.3863,2.3863,0.0000,0.0000,0.0000,0.0000,1.2877,...,0.0000,0.0000,0.0000,2.3863,2.3863,0.0000,4.7726,0.0000,0.0000,0.0000
D4,2.3863,0.0000,2.3863,0.0000,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,...,2.3863,0.0000,4.7726,0.0000,0.0000,2.3863,0.0000,2.3863,2.3863,0.0000


In [ ]:
# Contoh perhitungan manual
term_contoh = "jaringan"
dokumen_contoh = "D2"

tf = tf_manual.loc[dokumen_contoh, term_contoh]
df = df_manual[term_contoh]
idf = idf_manual[term_contoh]
tfidf = tf * idf

print("Contoh Perhitungan TF-IDF")
print("-------------------------")
print("Dokumen :", dokumen_contoh)
print("Term    :", term_contoh)
print("TF      :", tf)
print("DF      :", df)
print("N       :", N)
print("IDF     :", round(idf, 4))
print("TF-IDF  :", round(tfidf, 4))

Contoh Perhitungan TF-IDF
-------------------------
Dokumen : D2
Term    : jaringan
TF      : 2.0
DF      : 1
N       : 4
IDF     : 2.3863
TF-IDF  : 4.7726


In [ ]:
# Menggabungkan token menjadi teks kembali
dokumen_preprocessed = [
    " ".join(tokens)
    for tokens in df_preprocessing["Token"]
]

# Membuat TF-IDF menggunakan Scikit-Learn
vectorizer = TfidfVectorizer(
    lowercase=False,
    norm=None,
    smooth_idf=False
)

matrix_sklearn = vectorizer.fit_transform(dokumen_preprocessed)

# Mengambil daftar term dari Scikit-Learn
terms_sklearn = vectorizer.get_feature_names_out()

# Membuat DataFrame
df_tfidf_sklearn = pd.DataFrame(
    matrix_sklearn.toarray(),
    index=df_preprocessing["Dokumen"],
    columns=terms_sklearn
)

print("Matriks TF-IDF Scikit-Learn:")
display(df_tfidf_sklearn.round(4))

Matriks TF-IDF Scikit-Learn:


,bagian,banyak,bekerja,belajar,berkembang,bersama,bertukar,cepat,data,digunakan,...,menjalankan,pekerjaan,perangkat,sehari,semakin,sistem,teknologi,terdiri,tersebut,tugas
Dokumen,,,,,,,,,,,,,,,,,,,,,
D1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863,0.0000,1.2877,...,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863
D2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863,0.0000,2.3863,1.2877,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
D3,0.0000,2.3863,0.0000,2.3863,2.3863,0.0000,0.0000,0.0000,0.0000,1.2877,...,0.0000,0.0000,0.0000,2.3863,2.3863,0.0000,4.7726,0.0000,0.0000,0.0000
D4,2.3863,0.0000,2.3863,0.0000,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,...,2.3863,0.0000,4.7726,0.0000,0.0000,2.3863,0.0000,2.3863,2.3863,0.0000


In [ ]:
# Menghitung selisih antara TF-IDF manual dan Scikit-Learn
perbandingan = tfidf_manual - df_tfidf_sklearn

print("Selisih TF-IDF Manual dan Scikit-Learn:")
display(perbandingan.round(6))

# Mengecek selisih terbesar
selisih_maksimum = perbandingan.abs().max().max()

print("Selisih maksimum:", selisih_maksimum)

if selisih_maksimum < 0.000001:
    print("Hasil manual dan Scikit-Learn sama.")
else:
    print("Terdapat perbedaan hasil.")

Selisih TF-IDF Manual dan Scikit-Learn:


,bagian,banyak,bekerja,belajar,berkembang,bersama,bertukar,cepat,data,digunakan,...,menjalankan,pekerjaan,perangkat,sehari,semakin,sistem,teknologi,terdiri,tersebut,tugas
Dokumen,,,,,,,,,,,,,,,,,,,,,
D1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
D2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
D3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
D4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Selisih maksimum: 0.0
Hasil manual dan Scikit-Learn sama.


In [ ]:
print("===== TF-IDF MANUAL =====")
display(tfidf_manual.round(4))

print("\n===== TF-IDF SCIKIT-LEARN =====")
display(df_tfidf_sklearn.round(4))

===== TF-IDF MANUAL =====


,bagian,banyak,bekerja,belajar,berkembang,bersama,bertukar,cepat,data,digunakan,...,menjalankan,pekerjaan,perangkat,sehari,semakin,sistem,teknologi,terdiri,tersebut,tugas
Dokumen,,,,,,,,,,,,,,,,,,,,,
D1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863,0.0000,1.2877,...,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863
D2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863,0.0000,2.3863,1.2877,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
D3,0.0000,2.3863,0.0000,2.3863,2.3863,0.0000,0.0000,0.0000,0.0000,1.2877,...,0.0000,0.0000,0.0000,2.3863,2.3863,0.0000,4.7726,0.0000,0.0000,0.0000
D4,2.3863,0.0000,2.3863,0.0000,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,...,2.3863,0.0000,4.7726,0.0000,0.0000,2.3863,0.0000,2.3863,2.3863,0.0000



===== TF-IDF SCIKIT-LEARN =====


,bagian,banyak,bekerja,belajar,berkembang,bersama,bertukar,cepat,data,digunakan,...,menjalankan,pekerjaan,perangkat,sehari,semakin,sistem,teknologi,terdiri,tersebut,tugas
Dokumen,,,,,,,,,,,,,,,,,,,,,
D1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863,0.0000,1.2877,...,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863
D2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863,0.0000,2.3863,1.2877,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
D3,0.0000,2.3863,0.0000,2.3863,2.3863,0.0000,0.0000,0.0000,0.0000,1.2877,...,0.0000,0.0000,0.0000,2.3863,2.3863,0.0000,4.7726,0.0000,0.0000,0.0000
D4,2.3863,0.0000,2.3863,0.0000,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,...,2.3863,0.0000,4.7726,0.0000,0.0000,2.3863,0.0000,2.3863,2.3863,0.0000


In [ ]:
# Mencari term dengan nilai TF-IDF tertinggi pada setiap dokumen

print("Term dengan TF-IDF tertinggi pada setiap dokumen:")
print()

for dokumen_id in tfidf_manual.index:
    term_tertinggi = tfidf_manual.loc[dokumen_id].idxmax()
    nilai_tertinggi = tfidf_manual.loc[dokumen_id].max()

    print(
        f"{dokumen_id}: {term_tertinggi} "
        f"(TF-IDF = {nilai_tertinggi:.4f})"
    )

Term dengan TF-IDF tertinggi pada setiap dokumen:

D1: cepat (TF-IDF = 2.3863)
D2: jaringan (TF-IDF = 4.7726)
D3: teknologi (TF-IDF = 4.7726)
D4: perangkat (TF-IDF = 4.7726)


## Analisis

Berdasarkan hasil TF-IDF, setiap dokumen memiliki term yang paling menonjol, yaitu "cepat" pada D1, "jaringan" pada D2, "teknologi" pada D3, dan "perangkat" pada D4. Term tersebut memiliki nilai tinggi karena cukup sering muncul dalam dokumen dan lebih jarang muncul pada dokumen lainnya. Hasil perhitungan manual juga sama dengan hasil Scikit-Learn, sehingga perhitungan TF-IDF yang dilakukan sudah sesuai.